# Lesson 15 — Query Lifecycle & the Optimizer

> **Prerequisite:** Tiers 1–4, especially lesson 13 (execution plans). This lesson explains how plans are *chosen*.

## What you'll learn
- The four phases a query goes through: parse → bind → optimize → execute
- What statistics are and how histograms drive cardinality estimates
- How to read a histogram with `DBCC SHOW_STATISTICS`
- Inspecting the plan cache (`sys.dm_exec_cached_plans`, `sys.dm_exec_query_stats`)
- Why estimates diverge from actuals — and how stale stats cause bad plans

## Setup
Run `setup.sql` once. It creates the `lesson15` schema with an indexed copy of `SalesOrderHeader` and fresh full-scan statistics. Re-run to reset.

**Tooling note:** run the SQL cells in this notebook with Azure Data Studio connected to the mssql-learn container (attach the SQL kernel to your saved localhost connection). Several examples mention the actual execution plan (Ctrl+M) and estimated plan (Ctrl+L) — those graphical views are SSMS features; run the queries in ADS to see results and messages, and switch to SSMS to compare Estimated vs Actual rows on the plan operators.

In [ ]:
USE AdventureWorks2022;
GO

## Concepts

### The lifecycle
1. **Parse** — syntax check, parse tree.
2. **Bind (algebrize)** — resolve names and types into a logical tree.
3. **Optimize** — the cost-based optimizer enumerates plans and picks the cheapest, using **statistics** to estimate how many rows each operator will process.
4. **Execute** — run the plan; cache it keyed by the statement so it can be reused.

### Statistics & cardinality estimation
A statistics object carries a **histogram** (up to 200 steps) describing the distribution of a column's values. The optimizer reads it to estimate rows for a predicate (cardinality). Good estimates → good plans. `DBCC SHOW_STATISTICS` prints the header, density vector, and histogram.

### The plan cache
Compiled plans live in memory and are reused. `sys.dm_exec_cached_plans` shows plans and reuse counts; `sys.dm_exec_query_stats` aggregates runtime metrics (reads, CPU, executions) per cached statement.

### When estimates go wrong
If data changes a lot but statistics don't refresh, the optimizer plans for the *old* distribution — estimates fall far from actuals and it may pick a bad plan (wrong join type, too little memory). `UPDATE STATISTICS` refreshes them.

## Worked investigations

### Example 1 — Parsing only (no binding/optimization/execution)

`SET PARSEONLY` — parse phase in isolation.

In [ ]:
-- SET PARSEONLY checks syntax. A typo here raises a parse error; a valid query returns nothing.
SET PARSEONLY ON;
SELECT SalesOrderID FROM lesson15.SalesOrderHeader WHERE CustomerID = 11000;
SET PARSEONLY OFF;
GO

### Example 2 — Compile without executing (parse + bind + optimize, no execute)

`SET SHOWPLAN_TEXT` — compile to a plan without executing.

In [ ]:
-- SHOWPLAN_TEXT produces a plan without running. Use estimated plan in SSMS (Ctrl+L),
-- or the text form below. It must be the only statement in its batch (separated by GO).
SET SHOWPLAN_TEXT ON;
GO
SELECT SalesOrderID, OrderDate, TotalDue
FROM lesson15.SalesOrderHeader
WHERE CustomerID = 11000;
GO
SET SHOWPLAN_TEXT OFF;
GO

### Example 3 — Read a statistics histogram

`DBCC SHOW_STATISTICS` — read a histogram.

In [ ]:
-- The histogram is how the optimizer guesses how many rows a predicate returns.
DBCC SHOW_STATISTICS ('lesson15.SalesOrderHeader', 'IX_lesson15_SOH_CustomerID');
-- Three result sets: header (rows, updated), density vector, and the histogram steps.

### Example 4 — Estimated vs actual rows

Estimated vs actual rows in the plan.

In [ ]:
-- Turn on the actual plan in SSMS (Ctrl+M) and compare the Estimated vs Actual Number of Rows
-- on the Index Seek. With fresh FULLSCAN stats they should match closely.
SELECT SalesOrderID, OrderDate, TotalDue
FROM lesson15.SalesOrderHeader
WHERE CustomerID = 11000;

### Example 5 — Find cached plans and their reuse counts for this database

`sys.dm_exec_cached_plans` — reuse counts.

In [ ]:
SELECT
    cp.usecounts                       AS TimesReused,
    cp.objtype,
    SUBSTRING(st.text, 1, 120)         AS QueryTextStart
FROM sys.dm_exec_cached_plans      AS cp
CROSS APPLY sys.dm_exec_sql_text(cp.plan_handle) AS st
WHERE st.text LIKE '%lesson15.SalesOrderHeader%'
  AND st.text NOT LIKE '%dm_exec_cached_plans%'
ORDER BY cp.usecounts DESC;

### Example 6 — Aggregate runtime stats per query from the plan cache

`sys.dm_exec_query_stats` — per-query runtime stats.

In [ ]:
SELECT TOP 5
    qs.execution_count,
    qs.total_logical_reads,
    qs.total_logical_reads / qs.execution_count AS AvgLogicalReads,
    SUBSTRING(st.text, 1, 120)                  AS QueryTextStart
FROM sys.dm_exec_query_stats        AS qs
CROSS APPLY sys.dm_exec_sql_text(qs.sql_handle) AS st
WHERE st.text LIKE '%lesson15.SalesOrderHeader%'
  AND st.text NOT LIKE '%dm_exec_query_stats%'
ORDER BY qs.total_logical_reads DESC;

### Example 7 — Make statistics stale, watch the estimate diverge, then fix it

Stale-stats demo — make the estimate wrong, then fix it.

In [ ]:
-- Skew the distribution by reassigning many rows to CustomerID 11000 WITHOUT updating stats.
-- (UPDATE avoids the identity/PK conflict that re-inserting rows would cause: SalesOrderID is
--  an IDENTITY column copied by SELECT INTO and is the clustered PK.)
UPDATE TOP (10000) lesson15.SalesOrderHeader
SET CustomerID = 11000
WHERE CustomerID <> 11000;
GO

-- The optimizer still believes the OLD (small) row count for CustomerID = 11000.
-- Run with the actual plan (Ctrl+M) — Estimated rows will be far below Actual rows.
SELECT SalesOrderID, OrderDate FROM lesson15.SalesOrderHeader WHERE CustomerID = 11000;

-- Refresh statistics; the estimate now matches reality again.
UPDATE STATISTICS lesson15.SalesOrderHeader WITH FULLSCAN;
SELECT SalesOrderID, OrderDate FROM lesson15.SalesOrderHeader WHERE CustomerID = 11000;

## Common issues this explains
- Stale statistics causing wildly wrong estimates and bad plans.
- Why the same query can flip to a worse plan after a big data change.
- The cardinality-estimation root of most "why did it pick *that* plan?" questions.

## Pitfalls
- `SHOWPLAN_TEXT`/`SHOWPLAN_ALL` must be the only statement in the batch — separate with `GO`.
- Auto-update stats triggers on a row-change threshold; large tables can go stale between triggers.
- A cached plan compiled for one parameter value may be reused for a very different one — that is parameter sniffing (covered in lessons 13 and 18).

## Cheatsheet
See `cheatsheets/06-execution-plans.md`

## Exercises
Open `exercises.ipynb` and work them in order. Solutions in `exercises-solutions.ipynb`.